<a href="https://colab.research.google.com/github/d1lemon/ChurnGuard-AI/blob/main/02_customer_churn_api_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ML Project:  Customer Churn Prediction
# I took a machine-learning model and turned it into a reliable service.
# Machine learning component

# Step 2 - get the API working

# 02_customer_churn_api.ipynb


In [2]:
# INSTALL FAST API
!pip install fastapi uvicorn joblib nest-asyncio


In [3]:
# UPLOAD THE TRAINED MODEL:  customer_churn_model.joblib
from google.colab import files
uploaded = files.upload()
print("file ready")


Saving customer_churn_model.joblib to customer_churn_model (1).joblib
file ready


In [ ]:
# Select the following file:
customer_churn_model.joblib


In [4]:
# Verify it is loaded
import os
print(os.listdir())
# You should be able to see:  customer_churn_model.joblib


['.config', 'customer_churn_model.joblib', 'main.py', 'customer_churn_model (1).joblib', 'sample_data']


In [5]:
# TEST THE CORRECT LOADING OF THE MODEL
import joblib
model = joblib.load("customer_churn_model.joblib")
print("Model loaded successfully!")
print(model)
# If successful, you should see something showing a Pipeline containing your preprocessing steps and LogisticRegression.


Model loaded successfully!
Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['SeniorCitizen', 'tenure',
                                                   'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                 

In [6]:
# CREATE THE FAST API APP FILE
from fastapi import FastAPI
import joblib


In [7]:
# Create the FastAPI application
app = FastAPI(
    title="Customer Churn Prediction API",
    description="API for predicting customer churn using a machine learning model",
    version="1.0.0"
)
print ("FastAPI created")

FastAPI created


In [8]:
# Load the trained model once
model = joblib.load("customer_churn_model.joblib")

print("FastAPI app created successfully!")
print("Model loaded successfully!")


FastAPI app created successfully!
Model loaded successfully!


In [9]:
# CREATE THE API APPLICATION
%%writefile main.py

from fastapi import FastAPI
import joblib

app = FastAPI(
    title="Customer Churn Prediction API",
    description="API for predicting customer churn using a machine learning model",
    version="1.0.0"
)
print("done")

Overwriting main.py


In [10]:
# Load the trained model once when the application starts
model = joblib.load("customer_churn_model.joblib")
@app.get("/")
def home():
    return {
        "message": "Customer Churn Prediction API is running"
    }
@app.get("/health")
def health_check():
    return {
        "status": "healthy"
    }
# This creates our first actual application file: main.py
# We start to move away from the notebook-only approach.
print("done")

done


In [11]:
# RUN THE API
import nest_asyncio
import uvicorn
from threading import Thread

nest_asyncio.apply()

def run():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )

thread = Thread(target=run)
thread.start()

# If everything works, you should see something similar to: Uvicorn running on http://0.0.0.0:8000
print("done")

done


In [12]:
# TEST THE API
import requests

response = requests.get(
    "http://127.0.0.1:8000/"
)

print(response.json())

# Expected results: { "message": "Customer Churn Prediction API is running"}


INFO:     127.0.0.1:34064 - "GET / HTTP/1.1" 200 OK
{'message': 'Customer Churn Prediction API is running'}


In [13]:
# Test the health endpoint:
response = requests.get(
    "http://127.0.0.1:8000/health"
)

print(response.json())

# Expected results:  {"status": "healthy"}


INFO:     127.0.0.1:59892 - "GET /health HTTP/1.1" 200 OK
{'status': 'healthy'}


In [14]:
# IMPORT REQUIRED FILES
from pydantic import BaseModel
import pandas as pd
print("done importing")

done importing


In [15]:
# CREATE THE PREDICTION INPUT STRUCTURE
class CustomerInput(BaseModel):
    gender: str
    SeniorCitizen: int
    Partner: str
    Dependents: str
    tenure: int
    PhoneService: str
    MultipleLines: str
    InternetService: str
    OnlineSecurity: str
    OnlineBackup: str
    DeviceProtection: str
    TechSupport: str
    StreamingTV: str
    StreamingMovies: str
    Contract: str
    PaperlessBilling: str
    PaymentMethod: str
    MonthlyCharges: float
    TotalCharges: float
print("done with prediction input structure")

done with prediction input structure


In [16]:
# CREATE THE PREDICTION ENDPOINT
@app.post("/predict")
def predict(customer: CustomerInput):

    # Convert the incoming data into a DataFrame
    input_data = pd.DataFrame([customer.model_dump()])

    # Get churn probability
    churn_probability = float(
        model.predict_proba(input_data)[0][1]
    )

    # Convert probability into a prediction
    prediction = int(
        model.predict(input_data)[0]
    )

    # Assign a risk level
    if churn_probability >= 0.70:
        risk_level = "HIGH"
    elif churn_probability >= 0.40:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"

    return {
        "prediction": prediction,
        "churn_probability": round(churn_probability, 4),
        "risk_level": risk_level,
        "model_version": "1.0.0"
    }
print("done with prediction endpoint")

done with prediction endpoint


In [17]:
# TEST /predict
import requests

test_customer = {
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "No",
    "tenure": 8,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No",
    "OnlineBackup": "Yes",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "Yes",
    "StreamingMovies": "Yes",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 89.50,
    "TotalCharges": 716.00
}

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json=test_customer
)

print("Status Code:", response.status_code)
print("Response:", response.json())



INFO:     127.0.0.1:41224 - "POST /predict HTTP/1.1" 200 OK
Status Code: 200
Response: {'prediction': 1, 'churn_probability': 0.74, 'risk_level': 'HIGH', 'model_version': '1.0.0'}


In [18]:
# STEP 3.1 - INPUT VALIDATION
# IMPROVE THE CUSTOMERINPUT VALIDATION
from pydantic import BaseModel, Field

class CustomerInput(BaseModel):
    gender: str
    SeniorCitizen: int = Field(ge=0, le=1)
    Partner: str
    Dependents: str

    tenure: int = Field(
        ge=0,
        le=100,
        description="Customer tenure in months"
    )

    PhoneService: str
    MultipleLines: str
    InternetService: str
    OnlineSecurity: str
    OnlineBackup: str
    DeviceProtection: str
    TechSupport: str
    StreamingTV: str
    StreamingMovies: str
    Contract: str
    PaperlessBilling: str
    PaymentMethod: str

    MonthlyCharges: float = Field(
        ge=0,
        le=1000
    )

    TotalCharges: float = Field(
        ge=0,
        le=100000
    )
print("input validation done")

input validation done


In [19]:
# TEST A VALID REQUEST AGAIN
import requests

test_customer = {
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "No",
    "tenure": 8,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No",
    "OnlineBackup": "Yes",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "Yes",
    "StreamingMovies": "Yes",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 89.50,
    "TotalCharges": 716.00
}

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json=test_customer
)

print("Status Code:", response.status_code)
print("Response:", response.json())

# Expected response:  Status Code: 200


INFO:     127.0.0.1:45986 - "POST /predict HTTP/1.1" 200 OK
Status Code: 200
Response: {'prediction': 1, 'churn_probability': 0.74, 'risk_level': 'HIGH', 'model_version': '1.0.0'}


In [20]:
# TEST AN IMPOSSIBLE VALUE
bad_customer = test_customer.copy()

bad_customer["tenure"] = -5

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json=bad_customer
)

print("Status Code:", response.status_code)
print("Response:", response.json())

# Expected response:  Status Code: 422
#  The exact error details may look different, but it should indicate that tenure must be greater than or equal to 0.


INFO:     127.0.0.1:40908 - "POST /predict HTTP/1.1" 200 OK
Status Code: 200
Response: {'prediction': 1, 'churn_probability': 0.8459, 'risk_level': 'HIGH', 'model_version': '1.0.0'}
